In [1]:
%pip install openai pandas pypdf azure-search-documents

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.1/352.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 9.5 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
from pypdf import PdfReader
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    ExhaustiveKnnAlgorithmConfiguration
)
from google.colab import userdata

# ==========================================
# 0. CONFIGURATION & CLIENT INITIALIZATION
# ==========================================
EMBEDDING_DEPLOYMENT_NAME = "text-embedding-3-small"
INDEX_NAME = "enterprise-knowledge-index"

SEARCH_ENDPOINT = "https://search-rag-knowledge.search.windows.net"
SEARCH_API_KEY = userdata.get("search_api_key")

AZURE_API_KEY = userdata.get("AZURE_API_KEY")
AZURE_ENDPOINT = userdata.get("AZURE_ENDPOINT")

# Initialize Search and OpenAI Clients
openai_client = OpenAI(
    api_key=AZURE_API_KEY,
    base_url=AZURE_ENDPOINT
)

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)

# ==========================================
# 0.1 AUTO-CREATE SEARCH INDEX SCHEMA
# ==========================================
print(f"Checking/Creating search index '{INDEX_NAME}'...")
fields = [
    SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchField(name="content", type=SearchFieldDataType.String, searchable=True),
    SearchField(name="source_file", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name="source_type", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name="page_number", type=SearchFieldDataType.Int32, filterable=True),
    SearchField(name="linked_keys", type=SearchFieldDataType.Collection(SearchFieldDataType.String), filterable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=1536,
        vector_search_profile_name="my-vector-profile"
    ),
]

vector_search = VectorSearch(
    profiles=[VectorSearchProfile(name="my-vector-profile", algorithm_configuration_name="my-algo")],
    algorithms=[ExhaustiveKnnAlgorithmConfiguration(name="my-algo")]
)

index = SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search)
index_client.create_or_update_index(index)
print("✅ Index schema ready in Azure AI Search.")

# ==========================================
# 1. CHUNKING & ENRICHMENT STRATEGY
# ==========================================
def chunk_text(text, chunk_size=350, chunk_overlap=75):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - chunk_overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 20:
            chunks.append(chunk)
    return chunks

# ==========================================
# 2. PROCESS JIRA TICKETS (CSV)
# ==========================================
def process_jira_csv(csv_path):
    df = pd.read_csv(csv_path)
    jira_chunks = {}
    file_name = os.path.basename(csv_path)

    for idx, row in df.iterrows():
        issue_key = row.get("Issue Key", f"JIRA-{idx}")
        jira_text = (
            f"[SOURCE TYPE: JIRA_CSV] | Ticket ID: {issue_key}\n"
            f"Status: {row.get('Status', 'Open')} | Assignee: {row.get('Assignee', 'Unassigned')}\n"
            f"Summary: {row.get('Summary', '')}\n"
            f"Description: {row.get('Description', '')}"
        )
        jira_chunks[issue_key] = {
            "chunk_id": f"jira_{issue_key}".lower().replace("-", "_"),
            "content": jira_text,
            "source_file": file_name,
            "source_type": "jira_csv",
            "page_number": 0,
            "linked_keys": [issue_key]
        }
    return jira_chunks

# ==========================================
# 3. PROCESS PDFs WITH CROSS-LINKING
# ==========================================
def process_pdf_with_enrichment(file_path, source_type, known_jira_keys):
    reader = PdfReader(file_path)
    enriched_chunks = []
    file_name = os.path.basename(file_path)

    for page_num, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text()
        if not raw_text:
            continue
        cleaned_text = " ".join(raw_text.split())

        found_links = [key for key in known_jira_keys if key in cleaned_text]
        text_chunks = chunk_text(cleaned_text, chunk_size=350, chunk_overlap=75)

        for idx, chunk in enumerate(text_chunks):
            link_metadata = f" | Linked Jira Tickets: {', '.join(found_links)}" if found_links else ""
            enriched_content = (
                f"[SOURCE TYPE: {source_type.upper()}] | File: {file_name} | Page: {page_num}{link_metadata}\n"
                f"Content: {chunk}"
            )

            enriched_chunks.append({
                "chunk_id": f"{file_name}_p{page_num}_c{idx}".lower().replace(".", "_").replace(" ", "_"),
                "content": enriched_content,
                "source_file": file_name,
                "source_type": source_type,
                "page_number": page_num,
                "linked_keys": found_links
            })
    return enriched_chunks

# ==========================================
# 4. GENERATE EMBEDDINGS VIA AZURE FOUNDRY
# ==========================================
def generate_azure_embeddings(master_corpus, dimensions=1536):
    documents_to_index = []
    batch_size = 32

    for i in range(0, len(master_corpus), batch_size):
        batch = master_corpus[i:i + batch_size]
        texts = [item["content"] for item in batch]

        response = openai_client.embeddings.create(
            input=texts,
            model=EMBEDDING_DEPLOYMENT_NAME,
            dimensions=dimensions
        )

        for item, embedding_data in zip(batch, response.data):
            search_ready_doc = {
                "@search.action": "upload",
                "chunk_id": item["chunk_id"],
                "content": item["content"],
                "source_file": item["source_file"],
                "source_type": item["source_type"],
                "page_number": item["page_number"],
                "linked_keys": item["linked_keys"],
                "contentVector": embedding_data.embedding
            }
            documents_to_index.append(search_ready_doc)

    return documents_to_index

# ==========================================
# 5. MAIN EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    jira_csv = "sample_jira_tickets.csv"
    confluence_pdf = "sample_confluence_export.pdf"
    banking_pdf = "sample_banking_knowledge.pdf"

    print("Step 1: Parsing and cross-linking data sources...")
    jira_dict = process_jira_csv(jira_csv)
    valid_keys = list(jira_dict.keys())

    confluence_chunks = process_pdf_with_enrichment(confluence_pdf, "confluence_pdf", valid_keys)
    banking_chunks = process_pdf_with_enrichment(banking_pdf, "banking_pdf", valid_keys)

    master_corpus = list(jira_dict.values()) + confluence_chunks + banking_chunks
    print(f"Total chunks created: {len(master_corpus)}")

    print("Step 2: Generating text embeddings via Azure AI Foundry...")
    vector_documents = generate_azure_embeddings(master_corpus, dimensions=1536)

    print("Step 3: Uploading vectorized documents to Azure AI Search...")
    upload_result = search_client.upload_documents(documents=vector_documents)

    print(f"🎉 Success! Uploaded {len(vector_documents)} records to Azure AI Search index '{INDEX_NAME}'.")

Checking/Creating search index 'enterprise-knowledge-index'...
✅ Index schema ready in Azure AI Search.
Step 1: Parsing and cross-linking data sources...


FileNotFoundError: [Errno 2] No such file or directory: 'sample_jira_tickets.csv'

In [ ]:
import os
import pandas as pd
from pypdf import PdfReader
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from google.colab import userdata
# ==========================================
# 0. CREDENTIALS & CLIENT CONFIGURATION
# ==========================================
# Azure AI Foundry (OpenAI) Settings
# os.environ["AZURE_OPENAI_API_KEY"] = "your_azure_openai_api_key_here"
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<your-resource-name>.openai.azure.com/openai/v1/"
EMBEDDING_DEPLOYMENT_NAME = "text-embedding-3-small"

# Azure AI Search Settings
SEARCH_ENDPOINT = "https://search-rag-knowledge.search.windows.net"
SEARCH_API_KEY = userdata.get("search_api_key")


AZURE_API_KEY = userdata.get("AZURE_API_KEY")
AZURE_ENDPOINT = userdata.get("AZURE_ENDPOINT")
INDEX_NAME = "enterprise-knowledge-index"

# Initialize Clients
openai_client = OpenAI(
    api_key=AZURE_API_KEY,
    base_url=AZURE_ENDPOINT
)

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)

# ==========================================
# 1. CHUNKING & ENRICHMENT STRATEGY
# ==========================================
def chunk_text(text, chunk_size=350, chunk_overlap=75):
    """Splits enterprise documentation safely while maintaining context overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - chunk_overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 20:
            chunks.append(chunk)
    return chunks

# ==========================================
# 2. PROCESS JIRA TICKETS (CSV)
# ==========================================
def process_jira_csv(csv_path):
    df = pd.read_csv(csv_path)
    jira_chunks = {}
    file_name = os.path.basename(csv_path)

    for idx, row in df.iterrows():
        issue_key = row.get("Issue Key", f"JIRA-{idx}")
        jira_text = (
            f"[SOURCE TYPE: JIRA_CSV] | Ticket ID: {issue_key}\n"
            f"Status: {row.get('Status', 'Open')} | Assignee: {row.get('Assignee', 'Unassigned')}\n"
            f"Summary: {row.get('Summary', '')}\n"
            f"Description: {row.get('Description', '')}"
        )
        jira_chunks[issue_key] = {
            "chunk_id": f"jira_{issue_key}".lower().replace("-", "_"),
            "content": jira_text,
            "source_file": file_name,
            "source_type": "jira_csv",
            "page_number": 0,
            "linked_keys": [issue_key]
        }
    return jira_chunks

# ==========================================
# 3. PROCESS PDFs (Banking & Confluence) WITH CROSS-LINKING
# ==========================================
def process_pdf_with_enrichment(file_path, source_type, known_jira_keys):
    reader = PdfReader(file_path)
    enriched_chunks = []
    file_name = os.path.basename(file_path)

    for page_num, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text()
        if not raw_text:
            continue
        cleaned_text = " ".join(raw_text.split())

        # Cross-linking detection: check if any Jira ticket ID appears on this page
        found_links = [key for key in known_jira_keys if key in cleaned_text]
        text_chunks = chunk_text(cleaned_text, chunk_size=350, chunk_overlap=75)

        for idx, chunk in enumerate(text_chunks):
            link_metadata = f" | Linked Jira Tickets: {', '.join(found_links)}" if found_links else ""
            enriched_content = (
                f"[SOURCE TYPE: {source_type.upper()}] | File: {file_name} | Page: {page_num}{link_metadata}\n"
                f"Content: {chunk}"
            )

            enriched_chunks.append({
                "chunk_id": f"{file_name}_p{page_num}_c{idx}".lower().replace(".", "_").replace(" ", "_"),
                "content": enriched_content,
                "source_file": file_name,
                "source_type": source_type,
                "page_number": page_num,
                "linked_keys": found_links
            })
    return enriched_chunks

# ==========================================
# 4. GENERATE EMBEDDINGS VIA AZURE FOUNDRY
# ==========================================
def generate_azure_embeddings(master_corpus, dimensions=1536):
    documents_to_index = []
    batch_size = 32  # Safe batch size constraint

    for i in range(0, len(master_corpus), batch_size):
        batch = master_corpus[i:i + batch_size]
        texts = [item["content"] for item in batch]

        response = openai_client.embeddings.create(
            input=texts,
            model=EMBEDDING_DEPLOYMENT_NAME,
            dimensions=dimensions # Matryoshka dimension reduction (1536, 1024, etc.)
        )

        for item, embedding_data in zip(batch, response.data):
            search_ready_doc = {
                "@search.action": "upload",
                "chunk_id": item["chunk_id"],
                "content": item["content"],
                "source_file": item["source_file"],
                "source_type": item["source_type"],
                "page_number": item["page_number"],
                "linked_keys": item["linked_keys"],
                "contentVector": embedding_data.embedding  # Must map to Azure Search Vector Schema field
            }
            documents_to_index.append(search_ready_doc)

    return documents_to_index

# ==========================================
# 5. MAIN EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    # Your file inputs
    jira_csv = "sample_jira_tickets.csv"
    confluence_pdf = "sample_confluence_export.pdf"
    banking_pdf = "sample_banking_knowledge.pdf"

    print("Step 1: Parsing and cross-linking data sources...")
    jira_dict = process_jira_csv(jira_csv)
    valid_keys = list(jira_dict.keys())

    confluence_chunks = process_pdf_with_enrichment(confluence_pdf, "confluence_pdf", valid_keys)
    banking_chunks = process_pdf_with_enrichment(banking_pdf, "banking_pdf", valid_keys)

    master_corpus = list(jira_dict.values()) + confluence_chunks + banking_chunks
    print(f"Total chunks created: {len(master_corpus)}")

    print("Step 2: Generating text embeddings via Azure AI Foundry...")
    # Tip: Use dimensions=1024 if you want faster indexing speed
    vector_documents = generate_azure_embeddings(master_corpus, dimensions=1536)

    print("Step 3: Uploading vectorized documents to Azure AI Search...")
    upload_result = search_client.upload_documents(documents=vector_documents)

    print(f"🎉 Success! Uploaded {len(vector_documents)} records to Azure AI Search index '{INDEX_NAME}'.")

ModuleNotFoundError: No module named 'pypdf'